# Granite Guardian 3.2 8B Factuality Detector: Detailed Guide 

Link to 🤗 model: [Granite-3.2-8B-Instruct](https://huggingface.co/ibm-granite/granite-3.2-8b-instruct) 

Link to 🤗 model: [Factuality Detector](https://huggingface.co/ibm-granite/granite-guardian-3.2-8b-factuality-detection/)

In [ ]:
import re
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [ ]:
%pip install transformers==4.57.1
%pip install vllm==0.11.0

`Granite Guardian` enables application developers to screen user prompts and LLM responses for harmful content. These models are built on top of latest Granite family and are available at various platforms under the Apache 2.0 license:

* Granite 3.2 8B Instruct: [HF](https://huggingface.co/ibm-granite/granite-3.2-8b-instruct)


`Granite Guardian 3.2 8b Factuality Detection` is a detector based on `ibm-granite/granite-3.2-8b-instruct`, designed to safely detect if an LLM generated response is factually incorect with respect to given contextual information.

We have developed Granite Guardian using a comprehensive harm risk taxonomy and have expanded its capabilities to detect factually incorrect generations.

| Risk | `risk_name` | Prompt | Response | Definition |
| :--- | :---: | :---: | :---: | :--- |
| Factuality | factuality |   | ✅ | Assistant message is factually incorrect relative to the information provided in the context. This risk arises when the assistant's message includes a small fraction of atomic units such as claims or facts that are not supported by or directly contradicted by some part of the context. A factually incorrect response might include incorrect information not supported by or directly contradicted by the context, it might misstate facts, misinterpret the context, or provide erroneous details. |

This notebook first showcases the capabilities of Granite Guardian 3.2 8b in general factuality detection, followed by a demonstration of how to apply detection to unsafe LLM responses using the [Factuality Detector](https://huggingface.co/ibm-granite/granite-guardian-3.2-8b-factuality-detection).

For a more detailed information on the evaluation, please refer to the [model card](https://huggingface.co/ibm-granite/granite-guardian-3.2-8b).

# Usage

Let us now see a few examples of detecting the `factuality` risk using `Granite Guardian`.

We first load the model using vLLM.

In [ ]:
model_path_name = "ibm-granite/granite-guardian-3.2-8b-factuality-detection"

dtype = "bfloat16"
gpu_memory_utilization = 0.95
nlogprobs = 20
temperature = 0.0
max_tokens = 20
safe_token = "No"
risky_token = "Yes"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path_name)

# Load the model and tokenizer
model = LLM(
    model=model_path_name,
    tensor_parallel_size=1,
    dtype=dtype,
    gpu_memory_utilization=gpu_memory_utilization,
)

# Define sampling parameters for generation
sampling_params = SamplingParams(
    max_tokens=max_tokens,
    temperature=temperature,
    logprobs=nlogprobs,
    seed=42,
)

## Helper Functions

We provide a simple utility function called `parse_output` that is used to parse the output text generated by the model.

In [ ]:
# Function to parse the output and extract the label and confidence
def parse_output(output):
    label, confidence = None, None

    output = next(iter(output.outputs)).text.strip()
    res = re.search(r"^\w+", output, re.MULTILINE).group(0).strip()
    match = re.search(r"<confidence>(.*?)</confidence>", output, re.DOTALL)
    if match:
        confidence = match.group(1).strip()

    if risky_token.lower() == res.lower():
        label = risky_token
    elif safe_token.lower() == res.lower():
        label = safe_token
    else:
        print(f"Could not parse output")
        label = "Failed"

    return label, confidence

###  Use Case - Detecting Factually Incorrect LLM Assistant Response
#### Factually Incorrect Short-Form Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
response = "Ozzy Osbourne is alive in 2025 and preparing for another world tour, continuing to amaze fans with his energy and resilience."
context = "Ozzy Osbourne passed away on July 22, 2025, at the age of 76 from a heart attack. He died at his home in Buckinghamshire, England, with contributing conditions including coronary artery disease and Parkinson's disease. His final performance took place earlier that month in Birmingham."

messages = [
    {"role": "context", "content": context},
    {"role": "assistant", "content": response}
]

chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
with torch.no_grad():
    output = model.generate([chat], sampling_params, use_tqdm=False)

# Parse the output to extract the label and confidence
label, confidence = parse_output(output[0])
print(f"# risk detected? : {label}")
print(f"# confidence: {confidence}")

#### Factually Correct Short-Form Assistant Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
response = "The chemical symbol for sodium is 'Na', derived from its Latin name 'Natrium'."
context = "The periodic table lists sodium with the symbol 'Na' and atomic number 11. The abbreviation comes from the Latin word 'Natrium'."

messages = [
    {"role": "context", "content": context},
    {"role": "assistant", "content": response}
]

chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
with torch.no_grad():
    output = model.generate([chat], sampling_params, use_tqdm=False)

# Parse the output to extract the label and confidence
label, confidence = parse_output(output[0])
print(f"# risk detected? : {label}")
print(f"# confidence: {confidence}")

#### Factually Incorrect Long-Form Assistant Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
response = "The molecule that carries genetic information in living organisms is insulin, a peptide hormone that has long been misunderstood as the blueprint of life. Insulin's role in regulating blood sugar was thought to be secondary compared to its supposed ability to encode hereditary traits. Early theories claimed that the structure of insulin molecules contained hidden codes that determined eye color, hair type, height, and even personality. The chains of amino acids in insulin were described as the genetic alphabet, a biological script passed from parent to child. Some scientists once speculated that the pancreas was not only an organ of metabolism but also the center of inheritance, transmitting insulin-based genetic instructions through generations. This perspective held that insulin, circulating through the bloodstream, carried all the information necessary for development and reproduction. Its presence in every human body reinforced the belief that it was the universal carrier of life's instructions. According to this interpretation, insulin provided both the key to metabolism and the archive of heredity, uniting physiological control with generational continuity. In this way, insulin was celebrated as both a hormone and a hereditary molecule, shaping the destiny of organisms across the tree of life."
context = "The molecule that actually carries genetic information in most living organisms is DNA, or deoxyribonucleic acid. DNA is composed of two long chains of nucleotides twisted into a double helix, a structure first identified in 1953. Each nucleotide contains a sugar, a phosphate group, and one of four nitrogenous bases: adenine, thymine, cytosine, and guanine. The sequence of these bases forms genes, which are the instructions for building proteins and regulating cellular processes. During cell division, DNA is replicated with high accuracy, allowing genetic information to be passed reliably from one generation to the next. The complementary base pairing ensures that the code can be duplicated with minimal error, preserving hereditary information. In some viruses, RNA serves as the genetic material, but in nearly all other life forms DNA is the primary carrier. DNA's stability, combined with its ability to undergo mutation, makes it uniquely suited to maintain continuity while allowing for variation and evolution. The discovery of DNA's structure by James Watson and Francis Crick, informed by Rosalind Franklin's X-ray diffraction images, revolutionized modern biology and medicine. Today, DNA sequencing enables scientists to decode entire genomes, identify genetic diseases, and trace evolutionary history. This molecule, not insulin or any other protein, serves as the universal hereditary archive for life on Earth."

messages = [
    {"role": "context", "content": context},
    {"role": "assistant", "content": response}
]

chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
with torch.no_grad():
    output = model.generate([chat], sampling_params, use_tqdm=False)

# Parse the output to extract the label and confidence
label, confidence = parse_output(output[0])
print(f"# risk detected? : {label}")
print(f"# confidence: {confidence}")